In [1]:
import requests as rq
import pandas as pd
import datetime as dt
from time import sleep

In [2]:
with open('accessToken.txt',"r") as file:
    access_token = file.read()

In [3]:
url = "https://api.upstox.com/v2/user/profile"
headers = {
    'accept': 'application/json',
    'Api-Version': '2.0',
    'Authorization': f'Bearer {access_token}'}
response = rq.get(url, headers=headers)
print(response.json()['status'])

success


In [3]:
df = pd.read_csv("https://assets.upstox.com/market-quote/instruments/exchange/complete.csv.gz")

In [4]:
def filter_df(df,lot_size):
    df = df[(df['exchange'] == 'NSE_FO') &(df['instrument_type'] == 'OPTIDX') &(df['lot_size'] == lot_size)]
    df = df[df['expiry'] ==  min(df['expiry'].unique())]
    return df


In [5]:
BNDF = filter_df(df,15)
NFDF = filter_df(df,50)

In [6]:
NFDF[(NFDF['strike'] == 21400) & (NFDF['option_type'] == 'CE')]['instrument_key'].iloc[0]

'NSE_FO|46409'

In [14]:
def get_quotes(instrument):
	ltp_data = {}
	# url = 'https://api-v2.upstox.com/market-quote/quotes'
	url = 'https://api-v2.upstox.com/market-quote/ltp'
	# url = "https://api.upstox.com/v2/market-quote/ohlc"
	headers = {
		'accept': 'application/json',
		'Api-Version': '2.0',
		'Authorization': f'Bearer {access_token}'}
	params = {'symbol': instrument,'interval':'1d'}
	res = rq.get(url, headers=headers, params=params).json()
	# print('instrument',instrument)
	return res
ltp_data = get_quotes(list(BNDF['instrument_key']))
# ltp_data

In [25]:
ltp_data['data']

{'NSE_FO:BANKNIFTY2432048100PE': {'last_price': 1592.25,
  'instrument_token': 'NSE_FO|41490'},
 'NSE_FO:BANKNIFTY2432043800CE': {'last_price': 2597.8,
  'instrument_token': 'NSE_FO|41000'},
 'NSE_FO:BANKNIFTY2432044600CE': {'last_price': 2365.65,
  'instrument_token': 'NSE_FO|41095'},
 'NSE_FO:BANKNIFTY2432047300PE': {'last_price': 872.5,
  'instrument_token': 'NSE_FO|41408'},
 'NSE_FO:BANKNIFTY2432043000PE': {'last_price': 1.7,
  'instrument_token': 'NSE_FO|40971'},
 'NSE_FO:BANKNIFTY2432053500CE': {'last_price': 2.1,
  'instrument_token': 'NSE_FO|50708'},
 'NSE_FO:BANKNIFTY2432045400CE': {'last_price': 1178.6,
  'instrument_token': 'NSE_FO|41207'},
 'NSE_FO:BANKNIFTY2432050000CE': {'last_price': 3.15,
  'instrument_token': 'NSE_FO|41654'},
 'NSE_FO:BANKNIFTY2432048900CE': {'last_price': 8.65,
  'instrument_token': 'NSE_FO|41547'},
 'NSE_FO:BANKNIFTY2432050300PE': {'last_price': 3289.15,
  'instrument_token': 'NSE_FO|47652'},
 'NSE_FO:BANKNIFTY2432044900PE': {'last_price': 22.0,
  'i

In [26]:
def find_option(optionChain,near_val):
	call_symbol = {}
	put_symbol = {}
	trade_symbol = {}
	for i in range(5):
		try:
			ltp_data = get_quotes(optionChain)['data']
		except:
			sleep(0.5)
			ltp_data = get_quotes(optionChain)['data']
			pass
		for k, v in ltp_data.items():
			# Call Symbol
			if float(v['last_price']) <= near_val and k[-2:] == 'CE':
				call_symbol.update({v['instrument_token']: float(v['last_price'])})
			# Put Symbol
			if float(v['last_price']) <= near_val and k[-2:] == 'PE':
				put_symbol.update({v['instrument_token']: float(v['last_price'])})
		if call_symbol and put_symbol:
			ce_val = min(list(call_symbol.values()), key=lambda x: abs(x-near_val))
			pe_val = min(list(put_symbol.values()), key=lambda x: abs(x-near_val))
			for a, b in call_symbol.items():
				if b == ce_val:
					trade_symbol.update({a: b})
			for c, d in put_symbol.items():
				if d == pe_val:
					trade_symbol.update({c: d})
		if trade_symbol:
			return trade_symbol	
		else:
			sleep(1)
	return 'Symbol Not found'
find_option(225)

{'NSE_FO|41384': 200.6, 'NSE_FO|41326': 211.35}

In [23]:
BNDF[BNDF['tradingsymbol'] == 'BANKNIFTY2432046200PE']

,instrument_key,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,option_type,exchange
44893,NSE_FO|41326,41326.0,BANKNIFTY2432046200PE,NaN,224.25,2024-03-20,46200.0,0.05,15.0,OPTIDX,PE,NSE_FO


In [20]:
from_date =(dt.datetime.now() - dt.timedelta(days=4)).date()
to_date =dt.datetime.now().date()

In [32]:

url = f"https://api.upstox.com/v2/historical-candle/NSE_FO|41708/1minute/{to_date}/{from_date}"
# url = "https://api.upstox.com/v2/historical-candle/intraday/NSE_FO|41708/1minute"
headers = {
    'accept': 'application/json',
    'Api-Version': '2.0',
    'Authorization': f'Bearer {access_token}'}
# params = {'symbol': 'NSE_FO|41708','interval':'1minute','from_date':from_date,'to_date':to_date}
# params = {'symbol': 'NSE_FO|41708','interval':'1minute'}
resp = rq.get(url, headers=headers).json()
resp

{'status': 'success',
 'data': {'candles': [['2023-12-18T15:29:00+05:30',
    97.7,
    98.25,
    95,
    95.85,
    462900,
    5330950],
   ['2023-12-18T15:28:00+05:30', 100.4, 100.75, 97.15, 97.75, 309350, 5330950],
   ['2023-12-18T15:27:00+05:30', 98.2, 100.55, 97.75, 100.15, 523100, 5330950],
   ['2023-12-18T15:26:00+05:30', 94.95, 98.35, 92.95, 98.35, 538850, 5348600],
   ['2023-12-18T15:25:00+05:30', 99.65, 100.4, 94.4, 95, 533900, 5348600],
   ['2023-12-18T15:24:00+05:30', 103.75, 104.25, 98.2, 99.85, 534700, 5348600],
   ['2023-12-18T15:23:00+05:30', 103, 105.25, 102.35, 103.75, 329850, 5460450],
   ['2023-12-18T15:22:00+05:30', 109, 109.2, 103, 103.1, 581550, 5460450],
   ['2023-12-18T15:21:00+05:30', 110.6, 111.2, 108.9, 109.05, 260950, 5460450],
   ['2023-12-18T15:20:00+05:30',
    108.55,
    111.95,
    108.4,
    110.6,
    357450,
    5435850],
   ['2023-12-18T15:19:00+05:30', 111.4, 112, 108.2, 108.5, 275800, 5435850],
   ['2023-12-18T15:18:00+05:30', 110.2, 111.9, 10